# 🔌 Notebook: Model Context Protocol (MCP)

In this notebook we learn what MCP is, build a small MCP server, and connect an LLM to it.

## 📚 Sources

- [Model Context Protocol Specification](https://modelcontextprotocol.io/)
- [FastMCP Documentation](https://gofastmcp.com/)
- [Ollama: Tool Calling](https://docs.ollama.com/capabilities/tool-calling)

---

Good luck exploring MCP! 🤗

## What is MCP?

In the previous notebook, a "tool" was just a Python function living right there in our notebook, which we passed straight into `tools=[...]`. That's simple, but it doesn't scale well: if you have useful tools (a database lookup, a company API, a file search), you'd have to copy-paste that same Python code into every single project that wants to use them.

**MCP (Model Context Protocol)** is an open standard that solves this by splitting tools out into their own standalone **server**. A server exposes things over a standardized protocol — and any MCP-compatible **client** (a notebook, an app, an LLM-powered agent) can connect to it, discover what's available, and use it, without ever importing the server's code directly.

In short:

- **MCP server** — a small standalone program that exposes tools, and possibly more (this notebook's server lives in `mcp_server.py`).
- **MCP client** — connects to one or more servers and uses whatever they expose. This notebook *is* an MCP client.

An MCP server can expose three different kinds of things, not just tools:

- **Tools** — actions the model can invoke, like our library lookups below. This is what we focused on in the previous notebook.
- **Resources** — read-only data the client can fetch, like a file, a database record, or in our case the full library catalog. Think of this as the "knowledge" or "database" side of MCP.
- **Prompts** — reusable, parameterized prompt templates the server can hand out, so multiple clients share the same well-tested wording instead of everyone writing their own.

We'll build and use all three below. As before, we'll use `LLM_REASONING` (`gemma4:26b`), since it supports tool calling.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads LLM_HOST from a .env file in the project root (see notebook 03 / setup.md)

LLM_HOST = os.environ["LLM_HOST"]  # the IP address you got in the lecture
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"  # the reasoning MoE model - also supports tool calling

## 1. The MCP Server

Our example server is a small **university library catalog**. Its data lives in `dummy.py` — a made-up dataset with three related "tables": `AUTHORS`, `MEMBERS` (people who can borrow books), and `BOOKS` (which reference an author, and, if checked out, a member).

Let's look at the server itself, `mcp_server.py`:

In [2]:
print(open("mcp_server.py").read())

"""A small MCP server for a (fictional) university library catalog.

MCP (Model Context Protocol) servers expose "tools" - regular Python functions -
over a standardized protocol, so any MCP-compatible client (an app, a notebook,
an LLM agent, ...) can discover and call them, without needing to import this
file directly. This is the same idea as passing a Python function as a tool to
an LLM (see the previous notebook), just decoupled into its own process.

We use the `fastmcp` library, which turns a decorated function into an MCP tool
automatically - reading the type hints for the parameter types, and the
docstring for the tool description, exactly like the `ollama` package did for
local tools in the previous notebook.

This file is not meant to be run interactively in a terminal - see 06_mcp.ipynb
for how a client starts and talks to it.
"""

from fastmcp import FastMCP

from dummy import AUTHORS, BOOKS, MEMBERS

mcp = FastMCP("University Library")


def _author_name(author_id: int) -

### A Quick Aside: What is a `@decorator`?

You've seen `@mcp.tool()` written directly above a function definition — that `@something` syntax is called a **decorator**. A decorator is a function that takes another function as input and gives back a (possibly modified) function in its place. Writing:

```python
@mcp.tool()
def search_books(query: str) -> list[dict]:
    ...
```

is really just a shorthand for:

```python
def search_books(query: str) -> list[dict]:
    ...
search_books = mcp.tool()(search_books)
```

`mcp.tool()` doesn't change what `search_books` computes — it wraps it, registers it with the server as an available tool (reading its type hints and docstring along the way), and hands back a function you can still call completely normally, e.g. `search_books("algorithms")` still works exactly as before. Decorators are used all over Python for this kind of "register this" or "wrap this with extra behavior" pattern — you'll also see them for things like caching (`@functools.lru_cache`) or measuring how long a function takes to run.

A few things to notice:

- `mcp = FastMCP("University Library")` creates the server; `"University Library"` is just a display name.
- The `@mcp.tool()` decorator turns a plain Python function into an MCP tool — `fastmcp` reads the type hints and the docstring to build the tool's schema automatically. This is the exact same idea as passing a Python function directly to `tools=[...]` in the previous notebook, just now the function lives on the server side instead of in the client's code.
- `@mcp.resource("library://catalog")` and `@mcp.prompt()` work the same way, but register a *resource* and a *prompt* instead of a tool — more on those in Section 3.5.
- `_author_name()` and `_member_name()` are regular helper functions *without* any decorator — they're not exposed to clients at all, only used internally by the tools/resources above them.

### Starting the Server

Notice the very last line: `mcp.run(transport="stdio", show_banner=False)`. **`stdio`** means the server talks to whoever started it via standard input/output — the same channels a program normally uses for `print()` and keyboard input. This has an important consequence: **you don't start this server by running `python mcp_server.py` yourself in a terminal and expecting to type commands into it.** If you did, it would just sit there waiting silently — it's expecting to speak the MCP protocol over stdin/stdout with a *client*, not with a human. Instead, the client (our notebook, below) launches `mcp_server.py` as a subprocess automatically and talks to it that way.

`stdio` is the simplest transport for local development, since there's no networking or port to configure. If you instead wanted a persistent server that keeps running and can be reached over the network by multiple clients (even remotely), FastMCP also supports `transport="http"` — in that case you *would* start it yourself once with `python mcp_server.py`, and clients would connect to a URL instead of launching a subprocess.

## 2. Connecting as a Client

We use `fastmcp`'s own `Client` class to connect. Since `mcp_server.py` uses `stdio`, simply pointing `Client` at the file path is enough — it takes care of launching the subprocess and speaking the protocol for us.

In [3]:
from fastmcp import Client as MCPClient

mcp_client = MCPClient("mcp_server.py")

# Normally you would open a connection with "async with mcp_client:" for a single block of code.
# Here we open it manually instead, so the connection stays alive across the next several cells -
# we'll close it explicitly with mcp_client.close() at the end of the notebook.
await mcp_client.__aenter__()
print("Connected:", mcp_client.is_connected())

Connected: True


Now let's ask the server what tools it has. Notice we never imported anything from `mcp_server.py` - the client discovered the tools purely by talking to the running server process over the protocol.

In [4]:
tools = await mcp_client.list_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}")

search_books: Search the library catalog by title, subject, or author name.
get_book_details: Get the full catalog entry for a book by its ISBN, including who currently has it if it's checked out.
get_author_books: Get all books by a given author.


## 3. Calling Tools

`mcp_client.call_tool(name, arguments)` runs a tool on the server and gives us back a result object; `.data` holds the actual return value (already converted back into a Python list/dict for us).

In [5]:
result = await mcp_client.call_tool("search_books", {"query": "algorithms"})
print(result.data)

[{'isbn': '978-0-262-03384-8', 'title': 'Introduction to Algorithms', 'author': 'Thomas H. Cormen', 'subject': 'Computer Science', 'year': 2009, 'available': True}]


Let's chain calls together: search for a subject, then look up full details for one hit.

In [6]:
search_result = await mcp_client.call_tool("search_books", {"query": "biology"})
print("Search results:", search_result.data)

first_isbn = search_result.data[0]["isbn"]
details_result = await mcp_client.call_tool("get_book_details", {"isbn": first_isbn})
print("\nDetails:", details_result.data)

Search results: [{'isbn': '978-0-321-97362-8', 'title': 'Campbell Biology', 'author': 'Lisa A. Urry', 'subject': 'Biology', 'year': 2016, 'available': True}, {'isbn': '978-0-8153-4432-2', 'title': 'Molecular Biology of the Cell', 'author': 'Lisa A. Urry', 'subject': 'Biology', 'year': 2014, 'available': False}]

Details: {'isbn': '978-0-321-97362-8', 'title': 'Campbell Biology', 'author': 'Lisa A. Urry', 'subject': 'Biology', 'year': 2016, 'available': True}


In [7]:
author_result = await mcp_client.call_tool("get_author_books", {"author_name": "Cormen"})
print(author_result.data)

[{'isbn': '978-0-262-03384-8', 'title': 'Introduction to Algorithms', 'year': 2009}]


## 3.5 Resources and Prompts: Beyond Tools

Tools are actions with arguments the model chooses. **Resources** are simpler: read-only data sitting at a fixed address (a "URI", like `library://catalog`) — no arguments, you just fetch it. This is the natural place to expose "knowledge" a client might want as context, without wrapping it in an action first. Let's fetch our library's full catalog resource:

In [8]:
resources = await mcp_client.list_resources()
for r in resources:
    print(f"{r.uri} - {r.description}")

catalog = await mcp_client.read_resource("library://catalog")
print("\nContent:", catalog[0].text[:200], "...")

library://catalog - The full library catalog, with author names already resolved.

Content: [{"isbn": "978-0-262-03384-8", "title": "Introduction to Algorithms", "author": "Thomas H. Cormen"}, {"isbn": "978-0-13-468599-1", "title": "Artificial Intelligence: A Modern Approach", "author": "Stu ...


The result comes back as a JSON-encoded string (the exact same data our `search_books` tool returns, just without needing a search query first). Now let's look at the **prompt** `recommend_book`: instead of us writing out "Recommend a book about ___" ourselves, the server hands us the fully-formed wording — we just supply the parameter.

In [9]:
prompts = await mcp_client.list_prompts()
for p in prompts:
    print(f"{p.name} - {p.description}")

rendered = await mcp_client.get_prompt("recommend_book", {"subject": "physics"})
prompt_text = rendered.messages[0].content.text
print("\nRendered prompt:", prompt_text)

recommend_book - Generate a prompt asking for a book recommendation in a given subject.

Rendered prompt: Recommend one book about physics from the library catalog, and briefly say why.


This is just a plain string — you could send it straight to an LLM like any other prompt (we'll set up the LLM connection next, in Section 4).

## 4. Combining MCP Tools with an LLM

So far *we* decided which tool to call, by hand. But this is exactly the same situation as the previous notebook's agent loop — except now the tools come from an MCP server instead of local Python functions. To let `LLM_REASONING` decide, we just need to convert each MCP tool into the JSON-schema tool format `ollama.chat()` expects: `t.inputSchema` is already a JSON schema (FastMCP built it from the type hints), so there's no need to write anything by hand.

In [10]:
from ollama import Client as OllamaClient

ollama_client = OllamaClient(host=LLM_URL)

ollama_tools = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.inputSchema,
        },
    }
    for tool in tools
]

Now the familiar agent loop from the previous notebook — the only difference is that executing a tool call means `await mcp_client.call_tool(...)` instead of calling a local Python function directly.

In [11]:
messages = [{"role": "user", "content": "Is Deep Learning by Ian Goodfellow available in the library?"}]

while True:
    response = ollama_client.chat(model=LLM_REASONING, messages=messages, tools=ollama_tools, think=True)
    messages.append(response.message)
    print("Content:", response.message.content)

    if not response.message.tool_calls:
        break

    for tc in response.message.tool_calls:
        print(f"  Calling {tc.function.name}({tc.function.arguments})")
        result = await mcp_client.call_tool(tc.function.name, tc.function.arguments)
        print(f"  -> {result.data}")
        messages.append({"role": "tool", "tool_name": tc.function.name, "content": str(result.data)})

Content: 
  Calling search_books({'query': 'Deep Learning Ian Goodfellow'})
  -> []


Content: 
  Calling search_books({'query': 'Deep Learning'})
  -> [{'isbn': '978-0-262-04616-9', 'title': 'Deep Learning', 'author': 'Ian Goodfellow', 'subject': 'Computer Science', 'year': 2016, 'available': True}]


Content: Yes, "Deep Learning" by Ian Goodfellow is available in the library.


When you're done talking to the server, close the connection — this shuts down the `mcp_server.py` subprocess we launched back in Section 2.

In [12]:
await mcp_client.close()
print("Connected:", mcp_client.is_connected())

Connected: False


## Exercise: Ask About a Borrowed Book

Using the same pattern as Section 4, let `LLM_REASONING` answer: *"Who has 'Linear Algebra Done Right' checked out, and when is it due back?"*

Steps:
1. Open a new connection to the server (it was closed above).
2. Build the `ollama_tools` list again from `mcp_client.list_tools()`.
3. Run the same agent loop with the question above.
4. Don't forget to close the connection afterwards.

In [13]:
# Your code here...

<details>
<summary><b>Show solution</b></summary>

```python
mcp_client = MCPClient("mcp_server.py")
await mcp_client.__aenter__()

tools = await mcp_client.list_tools()
ollama_tools = [
    {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": tool.inputSchema,
        },
    }
    for tool in tools
]

messages = [{"role": "user", "content": "Who has 'Linear Algebra Done Right' checked out, and when is it due back?"}]

while True:
    response = ollama_client.chat(model=LLM_REASONING, messages=messages, tools=ollama_tools, think=True)
    messages.append(response.message)
    print("Content:", response.message.content)

    if not response.message.tool_calls:
        break

    for tc in response.message.tool_calls:
        print(f"  Calling {tc.function.name}({tc.function.arguments})")
        result = await mcp_client.call_tool(tc.function.name, tc.function.arguments)
        print(f"  -> {result.data}")
        messages.append({"role": "tool", "tool_name": tc.function.name, "content": str(result.data)})

await mcp_client.close()
```

</details>